In [7]:
import matplotlib
matplotlib.use('Agg')

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import json
import csv
from pathlib import Path

In [11]:
# Фиксация seed для воспроизводимости
SEED = 67
torch.manual_seed(SEED)
np.random.seed(SEED)

# Определение устройства
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Создание папок для артефактов
Path("artifacts/figures").mkdir(parents=True, exist_ok=True)

Using device: cpu


In [12]:
# Трансформации
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # mean и std для EMNIST
])

# Загрузка EMNIST
train_val_dataset = datasets.EMNIST(
    root='./data', 
    split='balanced', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = datasets.EMNIST(
    root='./data', 
    split='balanced', 
    train=False, 
    download=True, 
    transform=transform
)

# Разбиение train на train и val (80/20)
train_size = int(0.8 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    train_val_dataset, 
    [train_size, val_size], 
    generator=generator
)

# DataLoaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train size: {len(train_dataset)}")
print(f"Val size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

# Sanity check
x_sample, y_sample = next(iter(train_loader))
print(f"\nBatch shape: {x_sample.shape}")
print(f"Labels shape: {y_sample.shape}")
print(f"Value range: [{x_sample.min():.3f}, {x_sample.max():.3f}]")
print(f"Number of classes: {len(train_val_dataset.classes)}")

Train size: 90240
Val size: 22560
Test size: 18800

Batch shape: torch.Size([128, 1, 28, 28])
Labels shape: torch.Size([128])
Value range: [-0.424, 2.821]
Number of classes: 47


In [13]:
class MLP_Base(nn.Module):
    """Базовая MLP без регуляризации"""
    def __init__(self, input_size=784, hidden_sizes=[256, 128], num_classes=47):
        super().__init__()
        layers = []
        layers.append(nn.Flatten())
        
        # Первый скрытый слой
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())
        
        # Остальные скрытые слои
        for i in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i+1]))
            layers.append(nn.ReLU())
        
        # Выходной слой
        layers.append(nn.Linear(hidden_sizes[-1], num_classes))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)


class MLP_Dropout(nn.Module):
    """MLP с Dropout"""
    def __init__(self, input_size=784, hidden_sizes=[256, 128], num_classes=47, dropout_p=0.3):
        super().__init__()
        layers = []
        layers.append(nn.Flatten())
        
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_p))
        
        for i in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i+1]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_p))
        
        layers.append(nn.Linear(hidden_sizes[-1], num_classes))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)


class MLP_BatchNorm(nn.Module):
    """MLP с BatchNorm"""
    def __init__(self, input_size=784, hidden_sizes=[256, 128], num_classes=47):
        super().__init__()
        layers = []
        layers.append(nn.Flatten())
        
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.BatchNorm1d(hidden_sizes[0]))
        layers.append(nn.ReLU())
        
        for i in range(len(hidden_sizes) - 1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i+1]))
            layers.append(nn.BatchNorm1d(hidden_sizes[i+1]))
            layers.append(nn.ReLU())
        
        layers.append(nn.Linear(hidden_sizes[-1], num_classes))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

In [14]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Обучение на одной эпохе"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    """Оценка модели"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


class EarlyStopping:
    """Early stopping для предотвращения переобучения"""
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model = None
        
    def __call__(self, val_acc, model):
        score = val_acc
        
        if self.best_score is None:
            self.best_score = score
            self.best_model = model.state_dict().copy()
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model = model.state_dict().copy()
            self.counter = 0

In [15]:
def run_experiment(model, train_loader, val_loader, criterion, optimizer, 
                   num_epochs, device, early_stopping=None):
    """Запуск одного эксперимента"""
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    model = model.to(device)
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}% - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        if early_stopping is not None:
            early_stopping(val_acc, model)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                model.load_state_dict(early_stopping.best_model)
                break
    
    return history, model

In [16]:
print("E1: Base model (no Dropout, no BatchNorm)")

model_e1 = MLP_Base(input_size=784, hidden_sizes=[256, 128], num_classes=47)
criterion = nn.CrossEntropyLoss()
optimizer_e1 = optim.Adam(model_e1.parameters(), lr=0.001)

history_e1, model_e1 = run_experiment(
    model_e1, train_loader, val_loader, criterion, optimizer_e1,
    num_epochs=10, device=device
)

best_val_acc_e1 = max(history_e1['val_acc'])
best_val_loss_e1 = min(history_e1['val_loss'])
print(f"\nE1 Best Val Acc: {best_val_acc_e1:.2f}%")

E1: Base model (no Dropout, no BatchNorm)
Epoch 1/10 - Train Loss: 1.0473, Train Acc: 69.16% - Val Loss: 0.6809, Val Acc: 78.40%
Epoch 2/10 - Train Loss: 0.5943, Train Acc: 80.77% - Val Loss: 0.5710, Val Acc: 80.73%
Epoch 3/10 - Train Loss: 0.4916, Train Acc: 83.43% - Val Loss: 0.5211, Val Acc: 82.85%
Epoch 4/10 - Train Loss: 0.4369, Train Acc: 84.99% - Val Loss: 0.4974, Val Acc: 83.46%
Epoch 5/10 - Train Loss: 0.3961, Train Acc: 85.86% - Val Loss: 0.4996, Val Acc: 83.36%
Epoch 6/10 - Train Loss: 0.3663, Train Acc: 86.74% - Val Loss: 0.4815, Val Acc: 83.96%
Epoch 7/10 - Train Loss: 0.3414, Train Acc: 87.52% - Val Loss: 0.5016, Val Acc: 83.24%
Epoch 8/10 - Train Loss: 0.3178, Train Acc: 88.18% - Val Loss: 0.5103, Val Acc: 83.56%
Epoch 9/10 - Train Loss: 0.3013, Train Acc: 88.64% - Val Loss: 0.5032, Val Acc: 83.94%
Epoch 10/10 - Train Loss: 0.2824, Train Acc: 89.18% - Val Loss: 0.5227, Val Acc: 83.52%

E1 Best Val Acc: 83.96%


In [17]:
print("E2: Model with Dropout")

model_e2 = MLP_Dropout(input_size=784, hidden_sizes=[256, 128], num_classes=47, dropout_p=0.3)
criterion = nn.CrossEntropyLoss()
optimizer_e2 = optim.Adam(model_e2.parameters(), lr=0.001)

history_e2, model_e2 = run_experiment(
    model_e2, train_loader, val_loader, criterion, optimizer_e2,
    num_epochs=10, device=device
)

best_val_acc_e2 = max(history_e2['val_acc'])
best_val_loss_e2 = min(history_e2['val_loss'])
print(f"\nE2 Best Val Acc: {best_val_acc_e2:.2f}%")

E2: Model with Dropout
Epoch 1/10 - Train Loss: 1.3799, Train Acc: 59.62% - Val Loss: 0.7370, Val Acc: 76.84%
Epoch 2/10 - Train Loss: 0.8664, Train Acc: 72.87% - Val Loss: 0.6111, Val Acc: 79.80%
Epoch 3/10 - Train Loss: 0.7628, Train Acc: 75.52% - Val Loss: 0.5615, Val Acc: 81.49%
Epoch 4/10 - Train Loss: 0.7093, Train Acc: 77.10% - Val Loss: 0.5261, Val Acc: 82.24%
Epoch 5/10 - Train Loss: 0.6745, Train Acc: 78.05% - Val Loss: 0.5235, Val Acc: 81.88%
Epoch 6/10 - Train Loss: 0.6480, Train Acc: 78.48% - Val Loss: 0.4998, Val Acc: 83.13%
Epoch 7/10 - Train Loss: 0.6300, Train Acc: 79.15% - Val Loss: 0.4882, Val Acc: 83.25%
Epoch 8/10 - Train Loss: 0.6078, Train Acc: 79.69% - Val Loss: 0.4824, Val Acc: 83.50%
Epoch 9/10 - Train Loss: 0.5989, Train Acc: 80.06% - Val Loss: 0.4828, Val Acc: 83.78%
Epoch 10/10 - Train Loss: 0.5908, Train Acc: 80.03% - Val Loss: 0.4797, Val Acc: 83.81%

E2 Best Val Acc: 83.81%


In [ ]:
print("E3: Model with BatchNorm")

model_e3 = MLP_BatchNorm(input_size=784, hidden_sizes=[256, 128], num_classes=47)
criterion = nn.CrossEntropyLoss()
optimizer_e3 = optim.Adam(model_e3.parameters(), lr=0.001)

history_e3, model_e3 = run_experiment(
    model_e3, train_loader, val_loader, criterion, optimizer_e3,
    num_epochs=10, device=device
)

best_val_acc_e3 = max(history_e3['val_acc'])
best_val_loss_e3 = min(history_e3['val_loss'])
print(f"\nE3 Best Val Acc: {best_val_acc_e3:.2f}%")

E3: Model with BatchNorm


In [ ]:
print("E4: Best model + EarlyStopping")

# Выбираем лучший между E2 и E3
if best_val_acc_e2 > best_val_acc_e3:
    print("Using Dropout model as base")
    model_e4 = MLP_Dropout(input_size=784, hidden_sizes=[256, 128], num_classes=47, dropout_p=0.3)
    best_type = "Dropout"
else:
    print("Using BatchNorm model as base")
    model_e4 = MLP_BatchNorm(input_size=784, hidden_sizes=[256, 128], num_classes=47)
    best_type = "BatchNorm"

criterion = nn.CrossEntropyLoss()
optimizer_e4 = optim.Adam(model_e4.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=5)

history_e4, model_e4 = run_experiment(
    model_e4, train_loader, val_loader, criterion, optimizer_e4,
    num_epochs=20, device=device, early_stopping=early_stopping
)

best_val_acc_e4 = max(history_e4['val_acc'])
best_val_loss_e4 = min(history_e4['val_loss'])
print(f"\nE4 Best Val Acc: {best_val_acc_e4:.2f}%")

# Сохранение лучшей модели
torch.save(model_e4.state_dict(), 'artifacts/best_model.pt')
print("Best model saved to artifacts/best_model.pt")

In [ ]:
print("O1: LR too high")

model_o1 = MLP_Dropout(input_size=784, hidden_sizes=[256, 128], num_classes=47, dropout_p=0.3)
criterion = nn.CrossEntropyLoss()
optimizer_o1 = optim.Adam(model_o1.parameters(), lr=0.1)  # Слишком большой LR

history_o1, model_o1 = run_experiment(
    model_o1, train_loader, val_loader, criterion, optimizer_o1,
    num_epochs=8, device=device
)

In [ ]:
print("O2: LR too small")

model_o2 = MLP_Dropout(input_size=784, hidden_sizes=[256, 128], num_classes=47, dropout_p=0.3)
criterion = nn.CrossEntropyLoss()
optimizer_o2 = optim.Adam(model_o2.parameters(), lr=1e-5)  # Слишком маленький LR

history_o2, model_o2 = run_experiment(
    model_o2, train_loader, val_loader, criterion, optimizer_o2,
    num_epochs=8, device=device
)

In [ ]:
print("O3: SGD + momentum + weight decay")

model_o3 = MLP_Dropout(input_size=784, hidden_sizes=[256, 128], num_classes=47, dropout_p=0.3)
criterion = nn.CrossEntropyLoss()
optimizer_o3 = optim.SGD(model_o3.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

history_o3, model_o3 = run_experiment(
    model_o3, train_loader, val_loader, criterion, optimizer_o3,
    num_epochs=10, device=device
)

In [ ]:
print("Final evaluation on test")

test_loss, test_acc = evaluate(model_e4, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
plt.close('all')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_e4['train_loss'], label='Train Loss')
ax1.plot(history_e4['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('E4: Training and Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(history_e4['train_acc'], label='Train Acc')
ax2.plot(history_e4['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('E4: Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('artifacts/figures/curves_best.png', dpi=150, bbox_inches='tight')
plt.close('all')
print("Saved: curves_best.png")

In [ ]:
import matplotlib.pyplot as plt
plt.close('all')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_o1['train_loss'], label='Train Loss (LR=0.1)')
ax1.plot(history_o1['val_loss'], label='Val Loss (LR=0.1)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('O1: LR too high (0.1)')
ax1.legend()
ax1.grid(True)

ax2.plot(history_o2['train_loss'], label='Train Loss (LR=1e-5)')
ax2.plot(history_o2['val_loss'], label='Val Loss (LR=1e-5)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('O2: LR too small (1e-5)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('artifacts/figures/curves_lr_extremes.png', dpi=150, bbox_inches='tight')
plt.close('all')
print("Saved: curves_lr_extremes.png")

In [ ]:
# Сохранение результатов в runs.csv
results = [
    {
        'experiment_id': 'E1',
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': '256-128, ReLU, no Dropout, no BatchNorm',
        'optimizer': 'Adam',
        'lr': 0.001,
        'momentum': 0,
        'weight_decay': 0,
        'epochs_trained': len(history_e1['val_acc']),
        'best_val_accuracy': best_val_acc_e1,
        'best_val_loss': best_val_loss_e1
    },
    {
        'experiment_id': 'E2',
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': '256-128, ReLU, Dropout(0.3), no BatchNorm',
        'optimizer': 'Adam',
        'lr': 0.001,
        'momentum': 0,
        'weight_decay': 0,
        'epochs_trained': len(history_e2['val_acc']),
        'best_val_accuracy': best_val_acc_e2,
        'best_val_loss': best_val_loss_e2
    },
    {
        'experiment_id': 'E3',
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': '256-128, ReLU, no Dropout, BatchNorm',
        'optimizer': 'Adam',
        'lr': 0.001,
        'momentum': 0,
        'weight_decay': 0,
        'epochs_trained': len(history_e3['val_acc']),
        'best_val_accuracy': best_val_acc_e3,
        'best_val_loss': best_val_loss_e3
    },
    {
        'experiment_id': 'E4',
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': f'256-128, ReLU, {best_type}, EarlyStopping(patience=5)',
        'optimizer': 'Adam',
        'lr': 0.001,
        'momentum': 0,
        'weight_decay': 0,
        'epochs_trained': len(history_e4['val_acc']),
        'best_val_accuracy': best_val_acc_e4,
        'best_val_loss': best_val_loss_e4
    },
    {
        'experiment_id': 'O1',
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': '256-128, ReLU, Dropout(0.3)',
        'optimizer': 'Adam',
        'lr': 0.1,
        'momentum': 0,
        'weight_decay': 0,
        'epochs_trained': len(history_o1['val_acc']),
        'best_val_accuracy': max(history_o1['val_acc']),
        'best_val_loss': min(history_o1['val_loss'])
    },
    {
        'experiment_id': 'O2',
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': '256-128, ReLU, Dropout(0.3)',
        'optimizer': 'Adam',
        'lr': 1e-5,
        'momentum': 0,
        'weight_decay': 0,
        'epochs_trained': len(history_o2['val_acc']),
        'best_val_accuracy': max(history_o2['val_acc']),
        'best_val_loss': min(history_o2['val_loss'])
    },
    {
        'experiment_id': 'O3',
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': '256-128, ReLU, Dropout(0.3)',
        'optimizer': 'SGD',
        'lr': 0.01,
        'momentum': 0.9,
        'weight_decay': 1e-4,
        'epochs_trained': len(history_o3['val_acc']),
        'best_val_accuracy': max(history_o3['val_acc']),
        'best_val_loss': min(history_o3['val_loss'])
    }
]

# Запись в CSV
with open('artifacts/runs.csv', 'w', newline='') as csvfile:
    fieldnames = ['experiment_id', 'dataset', 'seed', 'model_summary', 'optimizer', 
                  'lr', 'momentum', 'weight_decay', 'epochs_trained', 
                  'best_val_accuracy', 'best_val_loss']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    for result in results:
        writer.writerow(result)

print("Results saved to artifacts/runs.csv")

In [ ]:
# Сохранение конфига лучшей модели
best_config = {
    'dataset': 'EMNIST',
    'split': 'balanced',
    'architecture': {
        'type': best_type,
        'input_size': 784,
        'hidden_sizes': [256, 128],
        'num_classes': 47,
        'activation': 'ReLU',
        'dropout_p': 0.3 if best_type == "Dropout" else None
    },
    'training': {
        'optimizer': 'Adam',
        'lr': 0.001,
        'batch_size': batch_size,
        'early_stopping_patience': 5,
        'seed': SEED
    },
    'results': {
        'best_val_accuracy': best_val_acc_e4,
        'best_val_loss': best_val_loss_e4,
        'test_accuracy': test_acc,
        'test_loss': test_loss
    }
}

with open('artifacts/best_config.json', 'w') as f:
    json.dump(best_config, f, indent=4)

print("Best config saved to artifacts/best_config.json")